In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsproysmartdata01";

In [0]:
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';


In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

## Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.sms_antiguo (
producto string,
fecha_envio timestamp,
usuarioSAC string,
numero string,
control string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/sms_antiguo"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.sms_nuevo (
cliente string,
producto string,
fecha_envio timestamp,
usuario_creacion STRING,
control string,
telefono_normalizado string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/sms_nuevo"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.catalogo_cliente_producto (
  id_cliente integer,
  id_producto integer,
  cliente string,
  producto string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/catalogo_cliente_producto"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.emails (
  received timestamp,
  senderaddress string,
  recipientaddress string,
  subject string,
  status string,
  messagetraceid string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/emails"

## Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.catalogo_transformed (
  id_cliente integer,
  id_producto integer,
  cliente string,
  producto string,
  ingestion_date timestamp,
  llave string
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/catalogo_transformed"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.sms_transformed (
  anio integer,
  mes integer,
  cliente string,
  producto string,
  fecha_envio timestamp,
  usuario_creacion STRING,
  numero string,
  control string,
  plataforma string,
  origen  string,
  periodo string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/sms_transformed"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.emails_transformed (
  anio integer,
  mes integer,
  cliente string,
  producto string,
  received timestamp,
  senderaddress string,
  recipientaddress string,
  subject string,
  status string,
  messagetraceid string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/emails_transformed"

## Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_sms (
  anio integer,
  mes integer,
  cliente string,
  producto string,
  usuario_creacion STRING,
  total_sms INTEGER,
  periodo string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_sms"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_email (
  anio integer,
  mes integer,
  cliente string,
  producto string,
  senderaddress STRING,
  entregado INTEGER,
  fallido INTEGER,
  total INTEGER
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_email"